In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np

from sympy import *
from sympy.abc import x

from transformers import BertTokenizer, BertTokenizerFast

from deepxube.domains.symbolic_regression import SymbolicState, SymbolicGoal, SymbolicRegression

C:\Users\Cale\miniconda3\envs\deepxube\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tokenizer setup

In [2]:
def normalize(expr):
    import re
    return " ".join(re.findall(r'\d+|[a-zA-Z]+|[\+\-\*/\^\=\(\)]', expr))

In [3]:
tokenizer = BertTokenizer.from_pretrained('./', lowercase=True)
# tokenizer.vocab

In [5]:
expr = 2*x + x**2
tokens = tokenizer.tokenize(normalize(str(expr)))
print(tokens)

encoded_tokens = tokenizer.encode(tokens)
print(encoded_tokens)

['x', '*', '*', '2', '+', '2', '*', 'x']
[[2, 17, 3], [2, 11, 3], [2, 11, 3], [2, 22, 3], [2, 9, 3], [2, 22, 3], [2, 11, 3], [2, 17, 3]]


## Testing to_np

In [6]:
start_state = SymbolicState(2*x + x**2)
goal = SymbolicGoal(
    xs=np.array([0, 1, 2, 3, 4]),
    ys=np.array([0, 1, 4, 9, 16]),
    tolerance=0
)

In [10]:
regr = SymbolicRegression(random_walk_length=5)
regr

In [11]:
states = regr.sample_start_states(5)
states[0].expr

In [17]:
def to_np(states: list[SymbolicState], goals: list[SymbolicGoal]) -> list[np.array]:
    tokens = [tokenizer.tokenize(str(s.expr)) for s in states]
    encoded_tokens = [tokenizer.encode(t) for t in tokens]
    
    return [
        encoded_tokens,
        goals[0].xs,
        [g.ys for g in goals]
    ]

In [18]:
start_state = SymbolicState(2*x + x**2)

goal = SymbolicGoal(
    xs=np.array([0, 1, 2, 3, 4]),
    ys=np.array([0, 1, 4, 9, 16]),
    tolerance=0
)

to_np([start_state], [goal])

[[[[2, 17, 3],
   [2, 11, 3],
   [2, 11, 3],
   [2, 22, 3],
   [2, 9, 3],
   [2, 22, 3],
   [2, 11, 3],
   [2, 17, 3]]],
 array([0, 1, 2, 3, 4]),
 [array([ 0,  1,  4,  9, 16])]]